# 🛡️ BorderWatch · Linear Algebra in Action

**This notebook practices every linear-algebra concept from Sessions 10 and 11 on real BorderWatch data** — the five parquet files produced by the generator (~2M rows total).

#### 📐 Exercise Structure

| Part | Topic | # Exercises |
|:---:|:---:|:---:|
| **A** | Vector operations | 5 |
| **B** | Matrices | 6 |
| **C** | Eigenvalues · PCA · SVD | 6 |
| **D** | End-to-End synthesis | 1 |

#### 🧮 Per-Exercise Structure (3 consecutive cells)

1. **📝 Question** — phrased in two versions:
   - **Part (a): by hand** on a sub-matrix / few rows of the real data (pencil-solvable)
   - **Part (b): in code** on **all** data (~2M rows, true computation)

2. **✏️ Hand solution** — full LaTeX solution to part (a)

3. **💻 Code** — runs part (b) on the full data + compares to the hand solution

> 💡 **Recommendation**: before peeking at the solution, try part (a) yourself with pencil and paper. Real understanding comes from the effort.

#### 📦 Requirements

The notebook assumes the five parquet files (~2,000,000 rows total) are in the same directory:

```
./manufacturers_calls.parquet
./supplier_logistics_sms.parquet
./supplier_procurement_email.parquet
./procurement_sales_chat.parquet
./waze_smuggler_movements.parquet
```

---


## 🔧 Setup — Load Real Data + Build Feature Matrix

This cell loads the five real parquet files and builds `_mfg_agg` — the aggregated per-phone feature matrix. It mirrors the loading cells in `Mexico_Cartel_DataSet_v10_3.ipynb` (Cells 6-7).

The primary file we work with: **L1 (manufacturers_calls.parquet)** with ~514,000 rows and 3,734 unique phones.

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import eig, norm, det, inv, matrix_rank, svd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import gc

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'


print('Loading real BorderWatch parquet files...')
mfg  = pd.read_parquet('./manufacturers_calls.parquet')
sms  = pd.read_parquet('./supplier_logistics_sms.parquet')
eml  = pd.read_parquet('./supplier_procurement_email.parquet')
cht  = pd.read_parquet('./procurement_sales_chat.parquet')
waze = pd.read_parquet('./waze_smuggler_movements.parquet')


mfg['call_time'] = pd.to_datetime(mfg['call_time'])
mfg['hour'] = mfg['call_time'].dt.hour
mfg['day_of_week'] = mfg['call_time'].dt.dayofweek
mfg['is_night'] = (mfg['hour'].isin(list(range(21, 24)) + list(range(0, 5)))).astype(int)
mfg['is_weekend'] = (mfg['day_of_week'] >= 5).astype(int)

print(f'  L1 (manufacturers_calls):       {len(mfg):>9,} rows × {mfg.shape[1]} cols')
print(f'  L2 (supplier_logistics_sms):    {len(sms):>9,} rows × {sms.shape[1]} cols')
print(f'  L3 (supplier_procurement_email):{len(eml):>9,} rows × {eml.shape[1]} cols')
print(f'  L4 (procurement_sales_chat):    {len(cht):>9,} rows × {cht.shape[1]} cols')
print(f'  L5 (waze_smuggler_movements):   {len(waze):>9,} rows × {waze.shape[1]} cols')
print(f'  TOTAL: {len(mfg)+len(sms)+len(eml)+len(cht)+len(waze):,} rows')


print('\nBuilding _mfg_agg feature matrix per phone...')
_mfg_agg = mfg.groupby('caller_phone').agg(
    total_calls=('call_id', 'count'),
    unique_partners=('receiver_phone', 'nunique'),
    night_ratio=('is_night', 'mean'),
    weekend_ratio=('is_weekend', 'mean'),
    avg_duration=('call_duration_sec', 'mean'),
    sim_swap_count=('caller_sim_id', 'nunique'),
).reset_index()


_null_sim = (mfg.assign(sim_null=mfg['caller_sim_id'].isna().astype(int))
             .groupby('caller_phone')['sim_null'].mean()
             .reset_index(name='null_sim_rate'))
_mfg_agg = _mfg_agg.merge(_null_sim, on='caller_phone', how='left')


_mfg_agg['is_burner'] = (_mfg_agg['unique_partners'] == 1).astype(int)

print(f'  _mfg_agg shape:  {_mfg_agg.shape}')
print(f'  burners detected (unique_partners=1): {_mfg_agg["is_burner"].sum()}')
print()
print('Sample (first 5 rows):')
print(_mfg_agg.head())


---

# 📘 Part A — Vector Operations (Session 10)

5 exercises on unit vectors: addition, subtraction, dot product, normalization, cosine.
Each exercise has a hand-version on real phone vectors + a code-version on all data.


## Exercise 1 — Basic Vector Operations · Average Burner Profile

In BorderWatch we often need to compute the **average profile of a group** — for example, "what is the typical burner profile?". This is vector addition + scalar division.

### Part (a) — by hand

Three real burners from the data (`unique_partners=1`), with 4 features each `[total_calls, night_ratio, weekend_ratio, avg_duration]`:

$$v_1 = [117,\ 0.769,\ 0.299,\ 863.7]$$
$$v_2 = [96,\ 0.708,\ 0.271,\ 955.2]$$
$$v_3 = [102,\ 0.765,\ 0.196,\ 930.7]$$

**Tasks:**
1. Compute the sum $s = v_1 + v_2 + v_3$ (element-wise addition).
2. Compute the average profile $\bar{v} = s / 3$ (scalar multiplication by $\frac{1}{3}$).
3. Compute the difference vector $d = v_1 - \bar{v}$ — how does $v_1$ differ from the mean?

### Part (b) — code on all data

Run the computation on **all** 371 burners in the data (`_mfg_agg[is_burner==1]`) with **all** 7 features. Display the average profile.


### ✏️ Hand Solution

**Step 1 — Vector sum:** element-wise addition

$$s = v_1 + v_2 + v_3$$

| Dim | $v_1$ | $v_2$ | $v_3$ | $s$ |
|---|---:|---:|---:|---:|
| total_calls | 117 | 96 | 102 | **315** |
| night_ratio | 0.769 | 0.708 | 0.765 | **2.242** |
| weekend_ratio | 0.299 | 0.271 | 0.196 | **0.766** |
| avg_duration | 863.7 | 955.2 | 930.7 | **2749.6** |

$$s = [315,\ 2.242,\ 0.766,\ 2749.6]$$

**Step 2 — Scalar division by 3** (multiplication by $\frac{1}{3}$):

$$\bar{v} = \frac{s}{3} = \left[\frac{315}{3},\ \frac{2.242}{3},\ \frac{0.766}{3},\ \frac{2749.6}{3}\right]$$

$$\bar{v} = [105,\ 0.747,\ 0.255,\ 916.5]$$

**Interpretation:** the average burner makes 105 calls, 75% concentrated at night, 26% on weekends, with an average call duration of 15 minutes.

**Step 3 — Difference vector:**

$$d = v_1 - \bar{v} = [117-105,\ 0.769-0.747,\ 0.299-0.255,\ 863.7-916.5]$$

$$d = [12,\ 0.022,\ 0.044,\ -52.8]$$

**Interpretation:** $v_1$ is more active than the mean (more calls, more night, more weekend) but its calls are shorter.

> 💡 **Critical lesson**: this is the first operation done before **any** PCA or K-Means — centering (`X - X.mean(0)`). The first output of `StandardScaler` is exactly $d$ for every vector.


In [ ]:


burners_df = _mfg_agg[_mfg_agg['is_burner'] == 1].copy()
feature_cols = ['total_calls', 'unique_partners', 'night_ratio', 'weekend_ratio',
                'avg_duration', 'sim_swap_count', 'null_sim_rate']
X_burners = burners_df[feature_cols].values

print(f'Total burners: {len(burners_df)}')
print(f'Feature matrix shape: X_burners.shape = {X_burners.shape}')
print()


s = X_burners.sum(axis=0)
v_bar = X_burners.mean(axis=0)

print('Average burner profile (v_bar) over ALL', len(burners_df), 'burners:')
for col, val in zip(feature_cols, v_bar):
    print(f'  {col:<20s}: {val:>10.4f}')

print()
print('Sum of all burner vectors (s):')
for col, val in zip(feature_cols, s):
    print(f'  {col:<20s}: {val:>10.4f}')


print('\nDifference vectors d_i = v_i - v_bar (first 3 burners):')
diffs = X_burners[:3] - v_bar
print('  Phone v_1 difference:', np.round(diffs[0], 3))
print('  Phone v_2 difference:', np.round(diffs[1], 3))
print('  Phone v_3 difference:', np.round(diffs[2], 3))


## Exercise 2 — Dot Product · Composite Suspicion Score

In Stage 9 of BorderWatch (Cell 80) the composite score is computed as $\text{score} = w \cdot x$ — the dot product of a weight vector $w$ with each phone's signal vector $x$.

### Part (a) — by hand

Given a real phone from the data: **`+52 7007239412`** (zone: Piedras Negras). From `_mfg_agg`:

$$x_{\text{phone}} = [117,\ 1,\ 0.769,\ 0.299,\ 863.7,\ 1,\ 0.128]$$

(corresponding to `[total_calls, unique_partners, night_ratio, weekend_ratio, avg_duration, sim_swap_count, null_sim_rate]`)

Given a weight vector that emphasizes suspicious signals (negative weights reduce, positive weights raise):

$$w = [-0.005,\ -2.0,\ +3.0,\ +1.0,\ -0.001,\ +0.1,\ +5.0]$$

**Task:** compute $\text{score} = w \cdot x$ in full, element-by-element. Verify the sign.

### Part (b) — code on all data

Compute the composite score for **all 3,734 phones** and rank the top 10 suspects.


### ✏️ Hand Solution

**Dot Product** = sum of element-wise products:

$$w \cdot x = \sum_{i=1}^{7} w_i \cdot x_i$$

| $i$ | Description | $w_i$ | $x_i$ | $w_i \cdot x_i$ |
|:---:|---|---:|---:|---:|
| 1 | total_calls | $-0.005$ | $117$ | $-0.585$ |
| 2 | unique_partners | $-2.0$ | $1$ | $-2.000$ |
| 3 | night_ratio | $+3.0$ | $0.769$ | $+2.307$ |
| 4 | weekend_ratio | $+1.0$ | $0.299$ | $+0.299$ |
| 5 | avg_duration | $-0.001$ | $863.7$ | $-0.864$ |
| 6 | sim_swap_count | $+0.1$ | $1$ | $+0.100$ |
| 7 | null_sim_rate | $+5.0$ | $0.128$ | $+0.640$ |

**Sum:**

$$\text{score} = -0.585 - 2.000 + 2.307 + 0.299 - 0.864 + 0.100 + 0.640$$

$$\boxed{\text{score} = -0.103}$$

**Interpretation:** the score is slightly below zero. Lowering `unique_partners=1` (weight $-2$) "cancels" the boost from `night_ratio=0.769` (contribution $+2.31$). There may be a design bias in the weights — `unique_partners` should arguably get a **positive** contribution (degree=1 is suspicious!) rather than negative.

**Corrected** $w_2 = -2.0 \to +2.0$:
$$\text{score}_{\text{corrected}} = -0.585 + 2.000 + 2.307 + 0.299 - 0.864 + 0.100 + 0.640 = \boxed{+3.897}$$

Now the score is clearly positive — the phone is suspicious.

> 💡 **The important lesson**: the dot product is **the heart** of the composite score, linear regression, SHAP, and PCA. Its power is that each signal becomes a numeric contribution, and the total of contributions = the decision.


In [ ]:


w = np.array([-0.005, -2.0, 3.0, 1.0, -0.001, 0.1, 5.0])
print('Weight vector w =', w)
print(f'Shape: {w.shape}')
print()


X_all = _mfg_agg[feature_cols].values
print(f'X_all shape: {X_all.shape}  ({len(X_all):,} phones × {len(feature_cols)} features)')


scores = X_all @ w


_mfg_agg['composite_score'] = scores

top10 = _mfg_agg.sort_values('composite_score', ascending=False).head(10)

print('\nTop 10 suspects by composite score:')
print(top10[['caller_phone', 'composite_score', 'is_burner',
             'unique_partners', 'night_ratio']].to_string(index=False))

print(f'\nScore range: min={scores.min():.3f},  max={scores.max():.3f},  mean={scores.mean():.3f}')


## Exercise 3 — Linear Combination · Building a Weighted Score

In BorderWatch we sometimes want to construct a **new feature** as a linear combination of existing ones. For example, a "burner suspicion index" could be:

$$y = \alpha_1 \cdot \text{is\_degree\_1} + \alpha_2 \cdot \text{night\_ratio} + \alpha_3 \cdot \text{null\_sim\_rate}$$

What PCA, regression, and SHAP do is exactly this: find optimal coefficients $\alpha_i$. In this exercise we compute a linear combination by hand.

### Part (a) — by hand

Given 2 features on 4 real phones from L1:

| Phone | night_ratio ($x_1$) | weekend_ratio ($x_2$) |
|---|---:|---:|
| `+52 7007239412` | 0.769 | 0.299 |
| `+52 7011837879` | 0.708 | 0.271 |
| `+52 7017340082` | 0.765 | 0.196 |
| `+52 7000563420` | 0.151 | 0.244 |

**Tasks:**

1. Given coefficients $\alpha = [2, 1]$, compute $y = \alpha_1 x_1 + \alpha_2 x_2$ by hand for all four phones.
2. Which phones are **in the span** of vector $[1,1]$? (i.e., $x_2 = x_1$, on the line through the origin) — answer: none exactly, but which is closest?
3. On a regular sheet of paper, sketch the two vectors that are **linearly independent**.

### Part (b) — code on all data

Apply the linear combination over **all 3,734 phones** in `_mfg_agg`. Plot a histogram.


### ✏️ Hand Solution

**Step 1 — Linear combination $y_i = 2 x_{1i} + 1 x_{2i}$:**

| Phone | $x_1$ | $x_2$ | $2 x_1 + x_2$ |
|---|---:|---:|---:|
| `+52 7007239412` | 0.769 | 0.299 | $2(0.769) + 0.299 = \boxed{1.837}$ |
| `+52 7011837879` | 0.708 | 0.271 | $2(0.708) + 0.271 = \boxed{1.687}$ |
| `+52 7017340082` | 0.765 | 0.196 | $2(0.765) + 0.196 = \boxed{1.726}$ |
| `+52 7000563420` | 0.151 | 0.244 | $2(0.151) + 0.244 = \boxed{0.546}$ |

**Step 2 — In the span of $[1,1]$ (i.e., $x_2 = x_1$):**

The "perfect" vector on the line $[1,1]$ satisfies $x_1 = x_2$. Closeness:

| Phone | $|x_1 - x_2|$ | Closest? |
|---|---:|:---:|
| `+52 7007239412` | $|0.769 - 0.299| = 0.470$ | far |
| `+52 7011837879` | $|0.708 - 0.271| = 0.437$ | far |
| `+52 7017340082` | $|0.765 - 0.196| = 0.569$ | far |
| `+52 7000563420` | $|0.151 - 0.244| = 0.093$ | ✓ **closest** |

The 4th phone is the closest to the line $[1,1]$ — meaning its night and weekend ratios are roughly equal (both low, ~0.2). This is a "balanced low-activity" phone — neither a night-burner nor a weekend-burner.

**Step 3 — Linear independence:**

Three vectors in 2D — by definition cannot all be independent (max 2 independent vectors in $\mathbb{R}^2$). Choosing 2 vectors: $v_1=[0.769, 0.299]$ and $v_4=[0.151, 0.244]$ are clearly independent (not on the same line through the origin). $v_1$ and $v_2 = [0.708, 0.271]$ are very close in direction — almost dependent (but not exactly).

> 💡 **The lesson**: every column in `X_all` is a vector. PCA finds the directions where vectors have maximum independence (linear independence = orthogonality). Multicollinearity = high linear dependence between columns → PCA detects it as small eigenvalues.


In [ ]:


alpha = np.array([2, 1])
print('Coefficient vector alpha =', alpha)


X_2d = _mfg_agg[['night_ratio', 'weekend_ratio']].values
print(f'X_2d shape: {X_2d.shape}')


y_all = X_2d @ alpha
print(f'y_all shape: {y_all.shape}  (linear combination per phone)')


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(y_all, bins=60, color='#1d3557', alpha=0.8, edgecolor='k')
axes[0].axvline(y_all.mean(), color='red', linestyle='--',
                label=f'mean = {y_all.mean():.3f}')
axes[0].set_xlabel('y = 2·night + 1·weekend')
axes[0].set_ylabel('# phones')
axes[0].set_title('Distribution of linear combination y on ALL 3,734 phones')
axes[0].legend()


is_burner_mask = _mfg_agg['is_burner'].values.astype(bool)
axes[1].scatter(X_2d[~is_burner_mask, 0], X_2d[~is_burner_mask, 1],
                s=10, alpha=0.5, label='legit (degree > 1)', color='#3498db')
axes[1].scatter(X_2d[is_burner_mask, 0], X_2d[is_burner_mask, 1],
                s=20, alpha=0.7, label='burner (degree = 1)', color='#E63946')

xx = np.linspace(0, 1, 100)
axes[1].plot(xx, xx, 'k--', alpha=0.6, label='span of [1,1] (y = x)')
axes[1].set_xlabel('night_ratio')
axes[1].set_ylabel('weekend_ratio')
axes[1].set_title('Phones in 2D feature space (with span of [1,1])')
axes[1].legend()
plt.tight_layout()
plt.show()


_mfg_agg['y_linear_comb'] = y_all
top5 = _mfg_agg.sort_values('y_linear_comb', ascending=False).head(5)
print('\nTop 5 phones by y = 2·night + 1·weekend:')
print(top5[['caller_phone', 'night_ratio', 'weekend_ratio',
            'y_linear_comb', 'is_burner']].to_string(index=False))


## Exercise 4 — Norms (L1, L2, L∞) · Distance Measures for Phones

Linear algebra defines several types of "vector length" — and each measures a different aspect of the phone.

| Norm | Formula | What it measures |
|:---:|:---:|---|
| **L1** | $\sum_i \|x_i\|$ | Total absolute distance — sum of all activities |
| **L2** | $\sqrt{\sum_i x_i^2}$ | Euclidean distance — "true" length |
| **L∞** | $\max_i \|x_i\|$ | The single dominant signal |

### Part (a) — by hand

A burner from the data with 4 features (`[total_calls, night_ratio, weekend_ratio, avg_duration]`):

$$x = [117,\ 0.769,\ 0.299,\ 863.7]$$

**Compute by hand:**

1. $\|x\|_1$ (L1)
2. $\|x\|_2$ (L2)
3. $\|x\|_\infty$ (L∞)

### Part (b) — code on all data

Compute the three norms for **all 3,734 phones** and analyze the relationship between them.


### ✏️ Hand Solution

$$x = [117,\ 0.769,\ 0.299,\ 863.7]$$

**L1 (Manhattan Norm)** = sum of absolute values:

$$\|x\|_1 = |117| + |0.769| + |0.299| + |863.7| = \boxed{981.768}$$

**L2 (Euclidean Norm)** = square root of sum of squares:

$$\|x\|_2 = \sqrt{117^2 + 0.769^2 + 0.299^2 + 863.7^2}$$

$$= \sqrt{13689 + 0.592 + 0.089 + 745977.7}$$

$$= \sqrt{759667.4} \approx \boxed{871.6}$$

**L∞ (Maximum Norm)** = the single largest value:

$$\|x\|_\infty = \max(117,\ 0.769,\ 0.299,\ 863.7) = \boxed{863.7}$$

**Comparison and interpretation:**

| Norm | Value | Dominated by |
|---|---:|---|
| L1 | 981.768 | All features combined |
| L2 | 871.6 | The two large ones: total_calls + avg_duration |
| L∞ | 863.7 | avg_duration only (largest single value) |

**The critical lesson:** in **non-normalized** features (different scales: 0-1 for ratios vs. ~1000 for duration), the norms get dominated by the largest-scale features. This is why **StandardScaler is necessary** before any geometric algorithm (PCA, K-Means, regression with regularization).

After standardization, each feature contributes equally to the norms — which is what we want.

> 💡 **Operational implication**: when measuring "how far is this phone from the average", we'd never use raw L1/L2 on un-scaled features. We always center + standardize first.


In [ ]:


print('Computing norms on raw (unscaled) features for all 3,734 phones...')


X_raw = _mfg_agg[feature_cols].values

l1_norms = np.abs(X_raw).sum(axis=1)
l2_norms = np.sqrt((X_raw ** 2).sum(axis=1))
linf_norms = np.abs(X_raw).max(axis=1)

print(f'  L1 norms:  mean={l1_norms.mean():>8.2f}, max={l1_norms.max():>8.2f}')
print(f'  L2 norms:  mean={l2_norms.mean():>8.2f}, max={l2_norms.max():>8.2f}')
print(f'  L∞ norms:  mean={linf_norms.mean():>8.2f}, max={linf_norms.max():>8.2f}')


print('\nNow standardizing features...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

l1_scaled = np.abs(X_scaled).sum(axis=1)
l2_scaled = np.sqrt((X_scaled ** 2).sum(axis=1))
linf_scaled = np.abs(X_scaled).max(axis=1)

print(f'  L1 norms (scaled):  mean={l1_scaled.mean():>8.4f}, max={l1_scaled.max():>8.4f}')
print(f'  L2 norms (scaled):  mean={l2_scaled.mean():>8.4f}, max={l2_scaled.max():>8.4f}')
print(f'  L∞ norms (scaled):  mean={linf_scaled.mean():>8.4f}, max={linf_scaled.max():>8.4f}')


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(l2_norms, bins=80, color='#1d3557', alpha=0.8)
axes[0].set_title('L2 norms (raw scale, dominated by avg_duration)')
axes[0].set_xlabel('||x||_2')
axes[0].set_ylabel('# phones')

axes[1].hist(l2_scaled, bins=80, color='#2A9D8F', alpha=0.8)
axes[1].axvline(l2_scaled.mean(), color='red', linestyle='--',
                label=f'mean = {l2_scaled.mean():.2f}')
axes[1].set_title('L2 norms (scaled — balanced features)')
axes[1].set_xlabel('||x_scaled||_2')
axes[1].legend()
plt.tight_layout()
plt.show()


_mfg_agg['l2_scaled'] = l2_scaled
top5_outliers = _mfg_agg.sort_values('l2_scaled', ascending=False).head(5)
print('\nTop 5 outlier phones by L2 (scaled features):')
print(top5_outliers[['caller_phone', 'l2_scaled', 'is_burner',
                     'total_calls', 'night_ratio', 'null_sim_rate']].to_string(index=False))


## Exercise 5 — Cosine Similarity · Pattern Mirroring Detection

Cosine measures the **angle** between two vectors, regardless of their magnitude. In BorderWatch we use it to detect **pattern mirroring**: pairs of phones that act in unison even if one makes more calls than the other.

$$\cos(\theta) = \frac{u \cdot v}{\|u\| \cdot \|v\|}$$

- $\cos \to 1$: nearly identical direction → suspect mirroring
- $\cos \to 0$: orthogonal → independent
- $\cos \to -1$: opposite directions

### Part (a) — by hand

Two real burners from the data:

$$u = [117,\ 0.769,\ 0.299,\ 863.7]$$
$$v = [96,\ 0.708,\ 0.271,\ 955.2]$$

**Tasks:**

1. Compute $u \cdot v$ (dot product)
2. Compute $\|u\|_2$ and $\|v\|_2$
3. Compute $\cos(\theta) = \frac{u \cdot v}{\|u\| \|v\|}$
4. Verify $\theta$ in degrees: $\theta = \arccos(\cos \theta)$

### Part (b) — code on all data

Compute the cosine similarity matrix between every pair of the 371 burners. Find the top 5 most-similar pairs (highest cosine).


### ✏️ Hand Solution

$$u = [117,\ 0.769,\ 0.299,\ 863.7]$$
$$v = [96,\ 0.708,\ 0.271,\ 955.2]$$

**Step 1 — Dot product $u \cdot v$:**

$$u \cdot v = 117 \cdot 96 + 0.769 \cdot 0.708 + 0.299 \cdot 0.271 + 863.7 \cdot 955.2$$

$$= 11232 + 0.545 + 0.081 + 825048.9$$

$$\boxed{u \cdot v \approx 836281.5}$$

**Step 2 — Norms:**

$$\|u\|_2 = \sqrt{117^2 + 0.769^2 + 0.299^2 + 863.7^2} = \sqrt{759667.4} \approx 871.6$$

$$\|v\|_2 = \sqrt{96^2 + 0.708^2 + 0.271^2 + 955.2^2} = \sqrt{921618.2} \approx 960.0$$

**Step 3 — Cosine:**

$$\cos(\theta) = \frac{836281.5}{871.6 \cdot 960.0} = \frac{836281.5}{836736} \approx \boxed{0.99946}$$

**Step 4 — Angle in degrees:**

$$\theta = \arccos(0.99946) \approx \boxed{1.89°}$$

**Interpretation:** the angle between the two phones' profiles is only ~2 degrees — essentially identical direction. They are **almost perfectly mirrored**.

But beware: the cosine is dominated by `avg_duration` (~863, 955), which is the largest feature in scale. After standardization, the cosine would more accurately reflect "mirroring across all features".

> 💡 **The lesson**: cosine is robust to scale **of vectors as a whole** (magnitude), but not to per-feature scale (feature dominance). For pattern mirroring, **always normalize first**.


In [ ]:


print('Computing cosine similarity for all 371 burners (after standardization)...')

burners_df_indexed = _mfg_agg[_mfg_agg['is_burner'] == 1].reset_index(drop=True)
X_burners_raw = burners_df_indexed[feature_cols].values


X_burners_scaled = StandardScaler().fit_transform(X_burners_raw)


norms = np.linalg.norm(X_burners_scaled, axis=1, keepdims=True)
norms_safe = np.where(norms == 0, 1, norms)
X_normed = X_burners_scaled / norms_safe


cos_matrix = X_normed @ X_normed.T

print(f'cos_matrix shape: {cos_matrix.shape}')


iu = np.triu_indices_from(cos_matrix, k=1)
pair_cosines = cos_matrix[iu]
pair_i, pair_j = iu

top5_idx = np.argsort(-pair_cosines)[:5]
print('\nTop 5 most-similar burner pairs (highest cosine):')
for rank, idx in enumerate(top5_idx, 1):
    i, j = pair_i[idx], pair_j[idx]
    c = pair_cosines[idx]
    angle_deg = np.degrees(np.arccos(np.clip(c, -1, 1)))
    p1 = burners_df_indexed.loc[i, 'caller_phone']
    p2 = burners_df_indexed.loc[j, 'caller_phone']
    print(f'  {rank}. {p1}  ↔  {p2}    cos={c:.4f}   angle={angle_deg:.2f}°')


fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cos_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='cosine similarity')
ax.set_title(f'Cosine similarity matrix between all {len(burners_df_indexed)} burners')
ax.set_xlabel('burner index')
ax.set_ylabel('burner index')
plt.tight_layout()
plt.show()


---

# 📗 Part B — Matrices (Session 11)

6 exercises on matrices: adjacency, matrix multiplication, transpose, rank, determinant, inverse. All on real network data.


## Exercise 6 — Adjacency Matrix · Detecting Burners from a Subgraph

The adjacency matrix is the most basic representation of a network. In BorderWatch it's the input to centrality calculations, community detection, and chain analysis.

### Part (a) — by hand

A sub-network of 4 phones from L1. Edges (calls between phones):

- $A \to B$ (5 calls)
- $B \to A$ (3 calls)
- $A \to C$ (1 call)
- $D \to A$ (1 call)

**Tasks:**

1. Build the binary adjacency matrix $A_{ij} = 1$ if there is at least one call from $i$ to $j$.
2. Compute the **out-degree** of each phone (sum of each row in $A$).
3. Which phone has **degree = 1**? (this is the burner candidate)

### Part (b) — code on all data

Build the binary adjacency for all 3,734 phones in `_mfg_agg`. Verify the burner count (`is_burner=1` ≡ `out_degree=1`).


### ✏️ Hand Solution

**Step 1 — Adjacency matrix** (binary, ignoring counts):

We have 4 phones. Index them: $A=0, B=1, C=2, D=3$.

Edges as $i \to j$:
- $(0, 1)$: $A \to B$
- $(1, 0)$: $B \to A$
- $(0, 2)$: $A \to C$
- $(3, 0)$: $D \to A$

$$A_{\text{adj}} = \begin{pmatrix}
0 & 1 & 1 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
1 & 0 & 0 & 0
\end{pmatrix}$$

**Step 2 — Out-degree** = sum of each row:

| Phone | Out-Degree |
|---|---:|
| A (row 0) | $0+1+1+0 = 2$ |
| B (row 1) | $1+0+0+0 = 1$ |
| C (row 2) | $0+0+0+0 = 0$ |
| D (row 3) | $1+0+0+0 = 1$ |

**Step 3 — Burner candidates (degree=1):**

- B and D both have out-degree = 1 — both are burner candidates.
- C has degree=0 (never initiated) — not a sender at all.
- A is not a burner (degree=2).

> 💡 **The critical lesson:** the burner-detection rule is **degree = 1 in the out-adjacency matrix**. This is exactly the *direct definition* of `unique_partners=1` in `_mfg_agg`. Algebraically, it's a property of the row-sum vector of $A$.


In [ ]:


print('Building binary adjacency for all phones (subset for memory)...')


_adj = (mfg.groupby(['caller_phone', 'receiver_phone']).size()
        .reset_index(name='n_calls')
        .assign(adj=1))


out_degree = (_adj.groupby('caller_phone')['receiver_phone']
              .nunique()
              .reset_index(name='out_degree'))


verification = _mfg_agg.merge(out_degree, on='caller_phone', how='left')
verification['out_degree'] = verification['out_degree'].fillna(0).astype(int)


diff = (verification['out_degree'] != verification['unique_partners']).sum()
print(f'Mismatches between out_degree and unique_partners: {diff}')
assert diff == 0, "Algebraic definition must match operational one"
print('  ✓ Algebraic burner definition = unique_partners = out_degree = 1')


burner_count = (verification['out_degree'] == 1).sum()
total = len(verification)
print(f'\nBurner count via adjacency degree=1: {burner_count} / {total} phones')

print('\nDegree distribution (first 10 values):')
deg_dist = verification['out_degree'].value_counts().sort_index().head(10)
print(deg_dist.to_string())


## Exercise 7 — Matrix Multiplication ($A^2$) · Chain Detection

The square of the adjacency matrix $A^2$ identifies **2-step paths**: $A^2_{ij}$ counts how many distinct paths of length 2 connect $i$ to $j$. This is the core of Stage 14 (Chain Detection): identify chain leakers.

$$A^2_{ij} = \sum_k A_{ik} \cdot A_{kj}$$

### Part (a) — by hand

Same 4×4 adjacency from Exercise 6:

$$A = \begin{pmatrix}
0 & 1 & 1 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
1 & 0 & 0 & 0
\end{pmatrix}$$

**Tasks:**

1. Compute $A^2$ by hand using the matrix multiplication formula.
2. Identify **2-step paths** from $A$ back to $A$ (entry $A^2[0,0]$).
3. In BorderWatch terms: what does $A^2[i,i] > 0$ mean? (Hint: ping-pong between two phones)

### Part (b) — code on all data

Compute $A^2$ on the burners-only subgraph. Find all phones with $A^2[i,i] > 0$ (ping-pong burners).


### ✏️ Hand Solution

**Step 1 — Compute $A^2 = A \cdot A$:**

For each entry $A^2_{ij}$, $A^2_{ij} = \sum_k A_{ik} \cdot A_{kj}$.

Row 0 of $A^2$:
- $A^2_{0,0} = A_{0,0} \cdot A_{0,0} + A_{0,1} \cdot A_{1,0} + A_{0,2} \cdot A_{2,0} + A_{0,3} \cdot A_{3,0}$
  $= 0 \cdot 0 + 1 \cdot 1 + 1 \cdot 0 + 0 \cdot 1 = \boxed{1}$
- $A^2_{0,1} = 0 \cdot 1 + 1 \cdot 0 + 1 \cdot 0 + 0 \cdot 0 = 0$
- $A^2_{0,2} = 0 \cdot 1 + 1 \cdot 0 + 1 \cdot 0 + 0 \cdot 0 = 0$
- $A^2_{0,3} = 0 \cdot 0 + 1 \cdot 0 + 1 \cdot 0 + 0 \cdot 0 = 0$

Row 1 of $A^2$:
- $A^2_{1,0} = 1 \cdot 0 + 0 \cdot 1 + 0 \cdot 0 + 0 \cdot 1 = 0$
- $A^2_{1,1} = 1 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 + 0 \cdot 0 = \boxed{1}$
- (other entries = 0)

Row 2 of $A^2$: all zeros (C has degree-0).

Row 3 of $A^2$:
- $A^2_{3,0} = 1 \cdot 0 + 0 \cdot 1 + 0 \cdot 0 + 0 \cdot 1 = 0$
- $A^2_{3,1} = 1 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 + 0 \cdot 0 = 1$
- $A^2_{3,2} = 1 \cdot 1 + 0 \cdot 0 + 0 \cdot 0 + 0 \cdot 0 = 1$
- $A^2_{3,3} = 0$

$$A^2 = \begin{pmatrix}
1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 \\
0 & 0 & 0 & 0 \\
0 & 1 & 1 & 0
\end{pmatrix}$$

**Step 2 — 2-step paths from $A$ back to $A$** = $A^2_{0,0} = 1$:

There is exactly one 2-step path $A \to ? \to A$. The intermediate is $B$ (we verify: $A \to B \to A$, since $A_{0,1}=1$ and $A_{1,0}=1$).

**Step 3 — Interpretation: $A^2_{i,i} > 0$ means a phone has a "ping-pong loop" — sends and is sent back.**

In BorderWatch this is the signature of **bidirectional burner pairs**: $A \to B \to A$ over a short period.

For chain detection (Stage 14), $A^3_{i,j}$ measures whether $i$ "reaches" $j$ via a chain leaker (intermediate phone), and $A^k_{i,i}$ identifies cycles of length $k$.

> 💡 **The lesson**: matrix multiplication is **counting paths** in a graph. This is the magic of NetworkX, of PageRank, and of every chain detection in BorderWatch.


In [ ]:


print('Computing A^2 on the burners-only subgraph...')


burner_phones = burners_df['caller_phone'].tolist()
burner_set = set(burner_phones)

b2b = _adj[(_adj['caller_phone'].isin(burner_set)) &
           (_adj['receiver_phone'].isin(burner_set))]
print(f'  Burner-to-burner edges: {len(b2b):,}')


sub_phones = sorted(set(b2b['caller_phone']) | set(b2b['receiver_phone']))
phone_to_idx = {ph: i for i, ph in enumerate(sub_phones)}
n_sub = len(sub_phones)
print(f'  Subgraph size: {n_sub} phones')


A_sub = np.zeros((n_sub, n_sub), dtype=int)
for _, r in b2b.iterrows():
    i, j = phone_to_idx[r['caller_phone']], phone_to_idx[r['receiver_phone']]
    A_sub[i, j] = 1


A2 = A_sub @ A_sub


diag = np.diag(A2)
n_pingpong = (diag > 0).sum()
print(f'\n  Phones with A^2[i,i] > 0 (ping-pong): {n_pingpong} / {n_sub}')

print('\nFirst 10 ping-pong burners:')
pp_indices = np.where(diag > 0)[0][:10]
for idx in pp_indices:
    print(f'  {sub_phones[idx]}    A^2[i,i] = {diag[idx]}')


fig, ax = plt.subplots(figsize=(7, 6))
n_show = min(50, n_sub)
ax.imshow(A2[:n_show, :n_show], cmap='Blues', aspect='auto', vmax=2)
ax.set_title(f'A² (first {n_show} burners) — diagonal = ping-pong loops')
ax.set_xlabel('burner j (col)')
ax.set_ylabel('burner i (row)')
plt.tight_layout()
plt.show()


## Exercise 8 — Transpose + Trace · Building a Covariance Matrix

The matrix transpose $X^T$ flips rows and columns. It's the building block of the covariance matrix:

$$C = \frac{1}{n-1} X_c^T X_c$$

where $X_c$ is the **centered** matrix (after subtracting the mean from each column).

### Part (a) — by hand

A 3×2 matrix of 3 burners × 2 features (`[night_ratio, weekend_ratio]`):

$$X = \begin{pmatrix}
0.769 & 0.299 \\
0.708 & 0.271 \\
0.765 & 0.196
\end{pmatrix}$$

**Tasks:**

1. Compute the column means $\bar{x} = [\bar{x}_1, \bar{x}_2]$.
2. Center the matrix: $X_c = X - \bar{x}$ (subtract column mean from each row).
3. Compute $X_c^T$ (transpose).
4. Compute $X_c^T X_c$ (covariance-base — not yet divided by $n-1$).
5. Compute the **trace** of $X_c^T X_c$ (sum of the diagonal).

### Part (b) — code on all data

Apply this on the full standardized feature matrix (3,734 × 7). Display the covariance matrix.


### ✏️ Hand Solution

$$X = \begin{pmatrix}
0.769 & 0.299 \\
0.708 & 0.271 \\
0.765 & 0.196
\end{pmatrix}$$

**Step 1 — Column means:**

$$\bar{x}_1 = \frac{0.769 + 0.708 + 0.765}{3} = \frac{2.242}{3} \approx 0.747$$

$$\bar{x}_2 = \frac{0.299 + 0.271 + 0.196}{3} = \frac{0.766}{3} \approx 0.255$$

$$\bar{x} = [0.747,\ 0.255]$$

**Step 2 — Centered matrix $X_c = X - \bar{x}$:**

$$X_c = \begin{pmatrix}
0.769-0.747 & 0.299-0.255 \\
0.708-0.747 & 0.271-0.255 \\
0.765-0.747 & 0.196-0.255
\end{pmatrix} = \begin{pmatrix}
0.022 & 0.044 \\
-0.039 & 0.016 \\
0.018 & -0.059
\end{pmatrix}$$

**Step 3 — Transpose $X_c^T$:**

$$X_c^T = \begin{pmatrix}
0.022 & -0.039 & 0.018 \\
0.044 & 0.016 & -0.059
\end{pmatrix}$$

**Step 4 — $X_c^T X_c$ (2×2 matrix):**

Entry $(1,1)$ = $\sum_i X_{c,i1}^2 = 0.022^2 + (-0.039)^2 + 0.018^2 = 0.000484 + 0.001521 + 0.000324 = 0.002329$

Entry $(2,2)$ = $\sum_i X_{c,i2}^2 = 0.044^2 + 0.016^2 + (-0.059)^2 = 0.001936 + 0.000256 + 0.003481 = 0.005673$

Entry $(1,2) = (2,1)$ = $\sum_i X_{c,i1} X_{c,i2} = 0.022 \cdot 0.044 + (-0.039) \cdot 0.016 + 0.018 \cdot (-0.059)$
  $= 0.000968 - 0.000624 - 0.001062 = -0.000718$

$$X_c^T X_c = \begin{pmatrix}
0.002329 & -0.000718 \\
-0.000718 & 0.005673
\end{pmatrix}$$

**Step 5 — Trace:**

$$\text{trace}(X_c^T X_c) = 0.002329 + 0.005673 = \boxed{0.008002}$$

**The trace = total variance** in the data.

> 💡 **The critical lesson**: $\text{trace}(X^T X) = \sum_i \lambda_i$ — the trace equals the sum of eigenvalues. This is **the foundation of EVR (Explained Variance Ratio) in PCA**.


In [ ]:


print('Building the covariance matrix on the full standardized feature matrix...')


X_full = _mfg_agg[feature_cols].values
scaler = StandardScaler()
X_full_scaled = scaler.fit_transform(X_full)

n = X_full_scaled.shape[0]
print(f'  X shape: {X_full_scaled.shape}  ({n} phones × {len(feature_cols)} features)')


X_c_T = X_full_scaled.T
print(f'  X^T shape: {X_c_T.shape}')


XTX = X_c_T @ X_full_scaled
print(f'  X^T·X shape: {XTX.shape}')


C = XTX / (n - 1)
print(f'\n  Covariance matrix C = X^T·X / (n-1):')
print(pd.DataFrame(C, index=feature_cols, columns=feature_cols).round(3))


trace = np.trace(XTX)
print(f'\n  trace(X^T·X) = {trace:.4f}')
print(f'  trace(C) = {np.trace(C):.4f}   (total variance)')
print()


fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(C, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='covariance')
ax.set_xticks(range(len(feature_cols)))
ax.set_yticks(range(len(feature_cols)))
ax.set_xticklabels(feature_cols, rotation=45, ha='right')
ax.set_yticklabels(feature_cols)
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        ax.text(j, i, f'{C[i,j]:.2f}', ha='center', va='center',
                color='white' if abs(C[i,j]) > 0.5 else 'black', fontsize=9)
ax.set_title(f'Covariance matrix (trace = {np.trace(C):.2f})')
plt.tight_layout()
plt.show()


## Exercise 9 — Rank · Multicollinearity Detection

The rank of a matrix is the number of **linearly independent** rows/columns. In BorderWatch we check rank to identify multicollinearity: features that "contain the same information".

If $\text{rank}(X) < \min(m,n)$, there is multicollinearity.

### Part (a) — by hand

A 3×3 feature matrix with **deliberately correlated columns**:

$$X = \begin{pmatrix}
1 & 2 & 3 \\
2 & 4 & 6 \\
3 & 6 & 9
\end{pmatrix}$$

**Tasks:**

1. Note that column 2 = 2 × column 1 and column 3 = 3 × column 1 (all columns are multiples of column 1).
2. What is the rank? (Hint: row reduction)
3. What problem will this matrix cause if we try $X^T X^{-1}$? (Hint: not invertible)

### Part (b) — code on all data

Compute the rank of the standardized feature matrix (3,734 × 7). Also test what happens when we add a deliberately collinear column.


### ✏️ Hand Solution

$$X = \begin{pmatrix}
1 & 2 & 3 \\
2 & 4 & 6 \\
3 & 6 & 9
\end{pmatrix}$$

**Step 1 — Recognize the structure:**

- Column 1: $[1, 2, 3]^T$
- Column 2: $[2, 4, 6]^T = 2 \times \text{Col 1}$
- Column 3: $[3, 6, 9]^T = 3 \times \text{Col 1}$

All three columns are linearly dependent — they all lie on the same line!

**Step 2 — Row reduction:**

Row 2: $R_2 - 2 R_1$:
$R_2 = [2,4,6] - 2 \cdot [1,2,3] = [0,0,0]$

Row 3: $R_3 - 3 R_1$:
$R_3 = [3,6,9] - 3 \cdot [1,2,3] = [0,0,0]$

Echelon form:

$$\text{rref}(X) = \begin{pmatrix}
1 & 2 & 3 \\
0 & 0 & 0 \\
0 & 0 & 0
\end{pmatrix}$$

Only **1 non-zero row** → $\boxed{\text{rank}(X) = 1}$.

**Step 3 — Inverse problem:**

Since $\text{rank}(X) = 1 < 3$, the determinant $\det(X) = 0$, and $X^{-1}$ does not exist. Worse: even $X^T X$ is **singular** (cannot be inverted) — so regression $\hat{\beta} = (X^T X)^{-1} X^T y$ fails.

> 💡 **The lesson**: in BorderWatch this is why we **never** use `unique_partners` together with `is_burner` (since `is_burner = (unique_partners==1)`). They are linearly dependent → rank drops → ML breaks.


In [ ]:


print('Computing the rank of the full feature matrix...')


print(f'\n1) Original feature matrix (after standardization):')
print(f'   X shape: {X_full_scaled.shape}')
rank_orig = matrix_rank(X_full_scaled)
print(f'   rank = {rank_orig}   (max possible = {min(X_full_scaled.shape)})')
print(f'   ✓ Full-rank — no multicollinearity')


print(f'\n2) Adding a deliberately collinear column (call_zscore = 2 × night_ratio + tiny noise)...')
X_with_collinear = np.column_stack([
    X_full_scaled,
    2 * X_full_scaled[:, 2] + 0.001 * np.random.randn(X_full_scaled.shape[0])
])
rank_collin = matrix_rank(X_with_collinear)
print(f'   X shape: {X_with_collinear.shape}')
print(f'   rank = {rank_collin}   (max possible = {min(X_with_collinear.shape)})')


print(f'\n3) Exact duplicate column (rank drops):')
X_with_dup = np.column_stack([
    X_full_scaled,
    X_full_scaled[:, 2]
])
rank_dup = matrix_rank(X_with_dup)
print(f'   X shape: {X_with_dup.shape}')
print(f'   rank = {rank_dup}   (max possible = {min(X_with_dup.shape)})')
print(f'   ✗ Rank dropped — features are linearly dependent')


print(f'\n4) Singular vs non-singular determinant test:')
print(f'   det(X^T X) original = {np.linalg.det(X_full_scaled.T @ X_full_scaled):.4e}')
print(f'   det(X^T X) with duplicate = {np.linalg.det(X_with_dup.T @ X_with_dup):.4e}   (≈ 0 → singular)')

print('\n   The duplicate column made X^T X singular and inversion would fail.')
print('   In BorderWatch terms: this prevents regression from running.')


## Exercise 10 — Determinant · Volume of Feature Space

The determinant of a matrix represents the **signed volume** of the parallelepiped formed by its rows/columns. In statistical applications it measures **information content**: a determinant near 0 means data lies on a flat subspace (low information).

### Part (a) — by hand

Two 2×2 matrices:

$$A = \begin{pmatrix} 3 & 1 \\ 2 & 4 \end{pmatrix}, \quad B = \begin{pmatrix} 1 & 2 \\ 2 & 4 \end{pmatrix}$$

**Tasks:**

1. Compute $\det(A) = a \cdot d - b \cdot c$.
2. Compute $\det(B)$. Note that the second row is twice the first.
3. Interpret: what does the geometry look like? (parallelogram area)

### Part (b) — code on all data

Compute the determinant of $X^T X$ for two cases:
- (i) The full feature matrix
- (ii) The matrix with one column duplicated

Compare the values.


### ✏️ Hand Solution

**Matrix $A$:**

$$\det(A) = (3)(4) - (1)(2) = 12 - 2 = \boxed{10}$$

**Matrix $B$:** (row 2 = 2 × row 1)

$$\det(B) = (1)(4) - (2)(2) = 4 - 4 = \boxed{0}$$

**Interpretation:**

- $A$: rows span a 2D parallelogram of area $10$. The 2 vectors are linearly independent.
- $B$: rows are collinear ($\vec{r_2} = 2\vec{r_1}$). They "span" a 1D line — a degenerate parallelogram with area 0.

**In BorderWatch terms:**

- $|\det(X^T X)| \gg 0$: features carry independent information. Healthy data → ML works.
- $|\det(X^T X)| \approx 0$: features are collinear. ML breaks. Use rank-detection (Exercise 9) or PCA to drop the redundant dimensions.

> 💡 **The lesson**: $\det = 0 \Leftrightarrow \text{singular} \Leftrightarrow$ inverse doesn't exist. PCA can "fix" this by projecting onto the non-degenerate subspace.


In [ ]:


print('Computing determinants...')


XTX_full = X_full_scaled.T @ X_full_scaled
det_full = np.linalg.det(XTX_full)
print(f'\n1) Original X^T·X ({XTX_full.shape}):')
print(f'   det = {det_full:.6e}')


X_with_dup_full = np.column_stack([X_full_scaled, X_full_scaled[:, 2]])
XTX_dup = X_with_dup_full.T @ X_with_dup_full
det_dup = np.linalg.det(XTX_dup)
print(f'\n2) X^T·X with duplicate column ({XTX_dup.shape}):')
print(f'   det = {det_dup:.6e}   (singular)')


X_corr_collin = np.column_stack([
    X_full_scaled,
    2 * X_full_scaled[:, 2] + 0.1 * np.random.randn(X_full_scaled.shape[0])
])
XTX_corr = X_corr_collin.T @ X_corr_collin
det_corr = np.linalg.det(XTX_corr)
print(f'\n3) X^T·X with near-collinear column (large noise):')
print(f'   det = {det_corr:.6e}')

X_corr_tight = np.column_stack([
    X_full_scaled,
    2 * X_full_scaled[:, 2] + 0.001 * np.random.randn(X_full_scaled.shape[0])
])
XTX_tight = X_corr_tight.T @ X_corr_tight
det_tight = np.linalg.det(XTX_tight)
print(f'\n4) X^T·X with near-collinear column (tiny noise):')
print(f'   det = {det_tight:.6e}   (much closer to 0)')


print(f'\nCondition number comparison:')
print(f'   cond(X^T·X) original    = {np.linalg.cond(XTX_full):.3e}')
print(f'   cond(X^T·X) tight collin = {np.linalg.cond(XTX_tight):.3e}   (worse — high collinearity)')

print('\nInterpretation:')
print('  As collinearity increases, det → 0, cond → ∞, and ML becomes unstable.')


## Exercise 11 — Matrix Inverse · Mahalanobis Distance

The Mahalanobis distance accounts for feature correlations:

$$d_M(x, \bar{x}) = \sqrt{(x - \bar{x})^T \Sigma^{-1} (x - \bar{x})}$$

where $\Sigma$ is the covariance matrix. This is **superior to Euclidean distance** for outlier detection in correlated data.

### Part (a) — by hand

Given a 2×2 covariance matrix:

$$\Sigma = \begin{pmatrix} 4 & 1 \\ 1 & 2 \end{pmatrix}$$

And a centered vector $z = x - \bar{x} = [2, 1]^T$.

**Tasks:**

1. Compute $\Sigma^{-1}$ using the 2×2 formula: $\Sigma^{-1} = \frac{1}{\det(\Sigma)} \begin{pmatrix} d & -b \\ -c & a \end{pmatrix}$.
2. Compute $\Sigma^{-1} z$.
3. Compute $z^T (\Sigma^{-1} z)$ — this is the squared Mahalanobis distance.
4. Take the square root → Mahalanobis distance $d_M$.

### Part (b) — code on all data

Compute Mahalanobis distance for each of the 3,734 phones against the mean. Identify the top-5 statistical outliers.


### ✏️ Hand Solution

$$\Sigma = \begin{pmatrix} 4 & 1 \\ 1 & 2 \end{pmatrix}, \quad z = \begin{pmatrix} 2 \\ 1 \end{pmatrix}$$

**Step 1 — Inverse:**

$\det(\Sigma) = (4)(2) - (1)(1) = 8 - 1 = 7$

$$\Sigma^{-1} = \frac{1}{7} \begin{pmatrix} 2 & -1 \\ -1 & 4 \end{pmatrix} = \begin{pmatrix} 2/7 & -1/7 \\ -1/7 & 4/7 \end{pmatrix}$$

**Step 2 — Compute $\Sigma^{-1} z$:**

$$\Sigma^{-1} z = \begin{pmatrix} 2/7 & -1/7 \\ -1/7 & 4/7 \end{pmatrix} \begin{pmatrix} 2 \\ 1 \end{pmatrix}$$

Row 1: $(2/7)(2) + (-1/7)(1) = 4/7 - 1/7 = 3/7$

Row 2: $(-1/7)(2) + (4/7)(1) = -2/7 + 4/7 = 2/7$

$$\Sigma^{-1} z = \begin{pmatrix} 3/7 \\ 2/7 \end{pmatrix}$$

**Step 3 — Compute $z^T \Sigma^{-1} z$ (squared distance):**

$$z^T (\Sigma^{-1} z) = (2)(3/7) + (1)(2/7) = 6/7 + 2/7 = 8/7 \approx 1.143$$

**Step 4 — Mahalanobis distance:**

$$d_M = \sqrt{8/7} \approx \boxed{1.069}$$

**Compare to Euclidean:**

$$d_{\text{Euclidean}} = \sqrt{2^2 + 1^2} = \sqrt{5} \approx 2.236$$

The Mahalanobis distance is **smaller** because the covariance $\Sigma_{1,1} = 4$ "knows" that the first feature naturally varies a lot — a deviation of 2 in feature 1 is small relative to its scale.

> 💡 **The lesson**: in BorderWatch, when we look for "outlier" phones, Mahalanobis distance is the **correct** metric, not Euclidean. It accounts for the natural variance of each feature and the correlation between features.


In [ ]:


print('Computing Mahalanobis distance for all 3,734 phones...')


C_full = np.cov(X_full_scaled.T)
print(f'  Covariance matrix shape: {C_full.shape}')


try:
    C_inv = np.linalg.inv(C_full)
    print('  ✓ C is invertible')
except np.linalg.LinAlgError:
    print('  ✗ C is singular — falling back to pseudo-inverse')
    C_inv = np.linalg.pinv(C_full)


mean_x = X_full_scaled.mean(axis=0)


Z = X_full_scaled - mean_x
mahal_squared = np.einsum('ij,jk,ik->i', Z, C_inv, Z)
mahal = np.sqrt(np.maximum(mahal_squared, 0))


euclidean = np.linalg.norm(Z, axis=1)


_mfg_agg['mahal_dist'] = mahal
_mfg_agg['euclidean_dist'] = euclidean

print('\nTop 5 outliers by Mahalanobis distance:')
top5_mahal = _mfg_agg.sort_values('mahal_dist', ascending=False).head(5)
print(top5_mahal[['caller_phone', 'mahal_dist', 'euclidean_dist',
                  'is_burner', 'total_calls']].to_string(index=False))

print('\nTop 5 outliers by Euclidean distance (for comparison):')
top5_euc = _mfg_agg.sort_values('euclidean_dist', ascending=False).head(5)
print(top5_euc[['caller_phone', 'mahal_dist', 'euclidean_dist',
                'is_burner', 'total_calls']].to_string(index=False))


fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(euclidean, mahal, alpha=0.4, s=6, c='#1d3557')
ax.set_xlabel('Euclidean distance')
ax.set_ylabel('Mahalanobis distance')
ax.set_title('Mahalanobis vs Euclidean distance (all 3,734 phones)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


---

# 📕 Part C — Eigenvalues, PCA, and SVD (Session 11)

6 exercises on the most powerful tools in linear algebra: eigenvalues, PCA, EVR, centrality, projection, and SVD. All applied to real BorderWatch data.


## Exercise 12 — Eigenvalues 2×2 · Variance per Axis

Eigenvalues of the covariance matrix $\Sigma$ tell us **how much variance** exists along each principal direction. This is the heart of PCA.

For a 2×2 symmetric matrix:

$$\Sigma = \begin{pmatrix} a & b \\ b & c \end{pmatrix}$$

The eigenvalues are roots of the characteristic polynomial:

$$\lambda^2 - (a+c)\lambda + (ac - b^2) = 0$$

$$\lambda = \frac{(a+c) \pm \sqrt{(a-c)^2 + 4b^2}}{2}$$

### Part (a) — by hand

A 2×2 covariance matrix from real BorderWatch data:

$$\Sigma = \begin{pmatrix}
0.001151 & -0.000337 \\
-0.000337 & 0.002835
\end{pmatrix}$$

**Tasks:**

1. Compute the two eigenvalues $\lambda_1, \lambda_2$.
2. Compute the eigenvector for $\lambda_1$.
3. What does this mean geometrically? (variance is highest along the direction of the largest eigenvector)

### Part (b) — code on all data

Compute eigenvalues and eigenvectors for the full 7×7 covariance matrix. Sort them in descending order — these are the PCA outputs.


### ✏️ Hand Solution

$$\Sigma = \begin{pmatrix}
0.001151 & -0.000337 \\
-0.000337 & 0.002835
\end{pmatrix}$$

Identify: $a = 0.001151$, $b = -0.000337$, $c = 0.002835$.

**Step 1 — Eigenvalues using the quadratic formula:**

Trace: $a + c = 0.003986$

Discriminant: $(a-c)^2 + 4b^2 = (0.001151 - 0.002835)^2 + 4(-0.000337)^2$
  $= (-0.001684)^2 + 4(0.0000001136)$
  $= 0.000002836 + 0.0000004544$
  $= 0.0000032902$

Square root: $\sqrt{0.0000032902} \approx 0.001814$

Eigenvalues:

$$\lambda_1 = \frac{0.003986 + 0.001814}{2} = \frac{0.005800}{2} = \boxed{0.002900}$$

$$\lambda_2 = \frac{0.003986 - 0.001814}{2} = \frac{0.002172}{2} = \boxed{0.001086}$$

**Step 2 — Eigenvector for $\lambda_1$:**

Solve $(\Sigma - \lambda_1 I)v = 0$:

$$\begin{pmatrix}
0.001151 - 0.002900 & -0.000337 \\
-0.000337 & 0.002835 - 0.002900
\end{pmatrix} v = 0$$

$$\begin{pmatrix}
-0.001749 & -0.000337 \\
-0.000337 & -0.0000650
\end{pmatrix} v = 0$$

From row 1: $-0.001749 v_1 - 0.000337 v_2 = 0 \Rightarrow v_2 = -5.19 v_1$

Choose $v_1 = 0.189$ (some scalar): $v_2 = -0.982$. Normalize so $\|v\| = 1$:

$$v_1 \approx \begin{pmatrix} 0.189 \\ -0.982 \end{pmatrix}$$

**Step 3 — Geometric interpretation:**

- $\lambda_1 \approx 0.0029$ is the variance along PC1 — the most important direction.
- $\lambda_2 \approx 0.0011$ is the variance along PC2 (orthogonal to PC1).
- Ratio $\lambda_1 / (\lambda_1 + \lambda_2) = 0.0029 / 0.0040 \approx 73\%$ — PC1 explains 73% of the variance.
- $v_1 \approx [0.19, -0.98]$ means PC1 is **mostly** "weekend_ratio" (component −0.98) with a small "night_ratio" component.

> 💡 **The critical lesson**: eigenvectors of $\Sigma$ are exactly the principal components of PCA. Eigenvalues tell you how important each direction is.


In [ ]:


print('Computing eigenvalues of the full 7x7 covariance matrix...')


C_full = np.cov(X_full_scaled.T)
print(f'  Covariance matrix shape: {C_full.shape}')


eigvals, eigvecs = eig(C_full)


eigvals = eigvals.real
eigvecs = eigvecs.real
order = np.argsort(-eigvals)
eigvals_sorted = eigvals[order]
eigvecs_sorted = eigvecs[:, order]

print('\nSorted eigenvalues (variance along each PC):')
for i, lam in enumerate(eigvals_sorted):
    print(f'  λ_{i+1} = {lam:.6f}')

total_var = eigvals_sorted.sum()
print(f'\nTotal variance (sum of eigenvalues): {total_var:.4f}')
print(f'   = trace(C):                       {np.trace(C_full):.4f}')


fig, axes = plt.subplots(1, 2, figsize=(13, 4))


axes[0].bar(range(1, len(eigvals_sorted)+1), eigvals_sorted,
            color='#1d3557', edgecolor='k')
axes[0].set_xticks(range(1, len(eigvals_sorted)+1))
axes[0].set_xticklabels([f'PC{i}' for i in range(1, len(eigvals_sorted)+1)])
axes[0].set_ylabel('Eigenvalue (variance)')
axes[0].set_title('Eigenvalues sorted (Scree Plot)')
axes[0].grid(alpha=0.3)


evr_cumulative = np.cumsum(eigvals_sorted) / total_var * 100
axes[1].plot(range(1, len(eigvals_sorted)+1), evr_cumulative,
             marker='o', color='#E63946', linewidth=2.5)
axes[1].axhline(y=90, color='gray', linestyle='--', label='90% threshold')
axes[1].set_xticks(range(1, len(eigvals_sorted)+1))
axes[1].set_xticklabels([f'PC{i}' for i in range(1, len(eigvals_sorted)+1)])
axes[1].set_ylabel('Cumulative variance explained (%)')
axes[1].set_title('Cumulative EVR')
axes[1].legend()
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0, 105)
plt.tight_layout()
plt.show()


print('\nFirst eigenvector (PC1) loadings:')
for col, val in zip(feature_cols, eigvecs_sorted[:, 0]):
    print(f'  {col:<20s}: {val:+.4f}')


## Exercise 13 — PCA from Scratch · Dimension Reduction

PCA in 4 steps:

1. **Center** the data: $X_c = X - \bar{x}$
2. **Compute covariance**: $C = \frac{1}{n-1} X_c^T X_c$
3. **Eigendecomposition**: find eigenvalues $\lambda_i$ and eigenvectors $v_i$ of $C$.
4. **Project**: $X_{\text{PCA}} = X_c \cdot V$, where $V$ contains the top $k$ eigenvectors as columns.

### Part (a) — by hand

The same 3×2 matrix from Exercise 8:

$$X_c = \begin{pmatrix}
0.022 & 0.044 \\
-0.039 & 0.016 \\
0.018 & -0.059
\end{pmatrix}$$

With eigenvalues from Exercise 12: $\lambda_1 = 0.0029$, $\lambda_2 = 0.00109$.

And the first eigenvector from Exercise 12:

$$v_1 = \begin{pmatrix} 0.189 \\ -0.982 \end{pmatrix}$$

**Tasks:**

1. Project all 3 burners onto PC1: $X_c \cdot v_1$ (a 3×1 vector).
2. Verify the variance of the projected scores equals $\lambda_1$.

### Part (b) — code on all data

Implement PCA from scratch on the full 7-feature data, and compare to `sklearn.decomposition.PCA`. Display the top 5 phones by PC1 score.


### ✏️ Hand Solution

$$X_c = \begin{pmatrix}
0.022 & 0.044 \\
-0.039 & 0.016 \\
0.018 & -0.059
\end{pmatrix}, \quad v_1 = \begin{pmatrix} 0.189 \\ -0.982 \end{pmatrix}$$

**Step 1 — Project all 3 burners onto PC1:**

Row 1: $z_1 = (0.022)(0.189) + (0.044)(-0.982) = 0.00416 - 0.0432 = -0.0390$

Row 2: $z_2 = (-0.039)(0.189) + (0.016)(-0.982) = -0.00737 - 0.0157 = -0.0231$

Row 3: $z_3 = (0.018)(0.189) + (-0.059)(-0.982) = 0.00340 + 0.0579 = +0.0613$

$$X_{\text{PCA}} = \begin{pmatrix} -0.0390 \\ -0.0231 \\ +0.0613 \end{pmatrix}$$

**Step 2 — Verify variance equals $\lambda_1$:**

Mean: $\bar{z} = (-0.0390 - 0.0231 + 0.0613) / 3 = -0.0008 / 3 \approx 0$ ✓ (centered)

Variance (sample, $/(n-1)$):

$$\text{var}(z) = \frac{(-0.0390 - 0)^2 + (-0.0231 - 0)^2 + (0.0613 - 0)^2}{2}$$

$$= \frac{0.001521 + 0.000534 + 0.003758}{2} = \frac{0.005813}{2} = 0.0029$$

$$\boxed{\text{var}(z) = 0.0029 = \lambda_1} \quad \checkmark$$

> 💡 **The critical lesson:** the variance of the PC1 projection equals exactly $\lambda_1$ — that's the entire point of PCA. We rotate the axes so the first axis captures the maximum possible variance.


In [ ]:


print('Implementing PCA from scratch + comparing to sklearn...')


X_centered = X_full_scaled - X_full_scaled.mean(axis=0)


C = np.cov(X_centered.T)


eigvals, eigvecs = eig(C)
eigvals = eigvals.real
eigvecs = eigvecs.real
order = np.argsort(-eigvals)
eigvals_sorted = eigvals[order]
eigvecs_sorted = eigvecs[:, order]


X_pca_manual = X_centered @ eigvecs_sorted


pca_sklearn = PCA(n_components=7)
X_pca_sklearn = pca_sklearn.fit_transform(X_centered)


print('\nEigenvalues comparison:')
print(f'   Manual:  {np.round(eigvals_sorted, 4)}')
print(f'   sklearn: {np.round(pca_sklearn.explained_variance_, 4)}   (variance per PC)')


pc1_manual_var = X_pca_manual[:, 0].var(ddof=1)
print(f'\nVariance of PC1 (manual):  {pc1_manual_var:.4f}')
print(f'Eigenvalue λ_1:             {eigvals_sorted[0]:.4f}')
print(f'  ✓ Variance(PC1) ≈ λ_1')


_mfg_agg['pc1_manual'] = X_pca_manual[:, 0]
_mfg_agg['pc2_manual'] = X_pca_manual[:, 1]

top5_pc1 = _mfg_agg.sort_values('pc1_manual', ascending=False).head(5)
print('\nTop 5 phones by PC1 score:')
print(top5_pc1[['caller_phone', 'pc1_manual', 'pc2_manual',
                'is_burner', 'unique_partners']].to_string(index=False))


fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_pca_manual[~is_burner_mask, 0], X_pca_manual[~is_burner_mask, 1],
           s=8, alpha=0.4, c='#3498db', label='legit')
ax.scatter(X_pca_manual[is_burner_mask, 0], X_pca_manual[is_burner_mask, 1],
           s=20, alpha=0.6, c='#E63946', label='burner')
ax.set_xlabel(f'PC1 ({eigvals_sorted[0]/eigvals_sorted.sum()*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({eigvals_sorted[1]/eigvals_sorted.sum()*100:.1f}% variance)')
ax.set_title('PCA projection of all 3,734 phones onto PC1–PC2')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Exercise 14 — EVR (Explained Variance Ratio) · Choosing Optimal k

EVR is the ratio of variance captured by each PC:

$$\text{EVR}_i = \frac{\lambda_i}{\sum_j \lambda_j} = \frac{\lambda_i}{\text{trace}(C)}$$

In practice, $k$ is chosen so that cumulative EVR $\geq 90\%$ (a common heuristic).

### Part (a) — by hand

Given eigenvalues from Exercise 12: $\lambda_1 = 0.002900$, $\lambda_2 = 0.001086$.

**Tasks:**

1. Total variance: $\text{trace}(C) = \lambda_1 + \lambda_2$.
2. EVR per PC: $\text{EVR}_i = \lambda_i / \text{trace}$.
3. Cumulative EVR: how much variance is explained by PC1 alone? By PC1+PC2?

### Part (b) — code on all data

Compute EVR for all 7 PCs. Find the smallest $k$ that captures ≥90% of variance.


### ✏️ Hand Solution

$$\lambda_1 = 0.002900, \quad \lambda_2 = 0.001086$$

**Step 1 — Total variance:**

$$\text{trace}(C) = 0.002900 + 0.001086 = 0.003986$$

**Step 2 — EVR per PC:**

$$\text{EVR}_1 = \frac{0.002900}{0.003986} \approx 0.7275 = \boxed{72.75\%}$$

$$\text{EVR}_2 = \frac{0.001086}{0.003986} \approx 0.2725 = \boxed{27.25\%}$$

**Step 3 — Cumulative EVR:**

| $k$ | $\sum_{i=1}^{k} \text{EVR}_i$ | Decision |
|:---:|:---:|---|
| 1 | 72.75% | Below 90% threshold — not enough |
| 2 | 100% | Above 90% — keep $k=2$ |

In this 2D toy case, $k=2$ "uses" all data. In real applications with 7+ features, we usually drop the last 2-3 PCs.

**A note on the 90% threshold:** there's no magic value; some research uses 80%, some 95%. It depends on:
- **Computation**: smaller $k$ → faster, less memory
- **Information**: less $k$ → may lose subtle patterns
- **Visualization**: $k=2$ is needed for 2D scatter plots, $k=3$ for 3D

> 💡 **The lesson:** EVR is the most important diagnostic in PCA — it tells you how much you "throw away" when you reduce dimensions.


In [ ]:


print('Computing EVR for the full 7-feature data...')


total_var = eigvals_sorted.sum()
evr_per_pc = eigvals_sorted / total_var
evr_cumulative = np.cumsum(evr_per_pc)

print('\nEVR table:')
print(f'  {"PC":<5}{"Eigenvalue":>12}{"EVR":>10}{"Cumulative":>14}')
print('  ' + '-' * 41)
for i, (lam, evr, cum) in enumerate(zip(eigvals_sorted, evr_per_pc, evr_cumulative)):
    marker = '  ← cross 90%' if cum >= 0.9 and (i == 0 or evr_cumulative[i-1] < 0.9) else ''
    print(f'  PC{i+1:<3} {lam:>12.5f} {evr*100:>9.2f}% {cum*100:>12.2f}%{marker}')


k_optimal = (evr_cumulative < 0.9).sum() + 1
print(f'\n✓ Optimal k = {k_optimal}  (captures {evr_cumulative[k_optimal-1]*100:.1f}% of variance)')


fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(1, len(evr_per_pc)+1), evr_per_pc * 100,
              color='#1d3557', edgecolor='k', alpha=0.85, label='EVR per PC')
ax_t = ax.twinx()
ax_t.plot(range(1, len(evr_cumulative)+1), evr_cumulative * 100,
          marker='o', color='#E63946', linewidth=2.5, label='Cumulative EVR')
ax_t.axhline(90, color='gray', linestyle='--', alpha=0.6)
ax_t.text(7.2, 91, '90% threshold', color='gray', fontsize=9)
ax_t.axvline(k_optimal + 0.5, color='#2A9D8F', linestyle=':', linewidth=2)
ax_t.text(k_optimal + 0.7, 75, f'k = {k_optimal}', color='#2A9D8F', fontsize=12, fontweight='bold')

ax.set_xlabel('Principal Component')
ax.set_ylabel('EVR per PC (%)', color='#1d3557')
ax_t.set_ylabel('Cumulative EVR (%)', color='#E63946')
ax.set_title('PCA Scree Plot — choosing optimal k')
ax.set_xticks(range(1, len(evr_per_pc)+1))
ax.set_xticklabels([f'PC{i}' for i in range(1, len(evr_per_pc)+1)])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Exercise 15 — Eigenvector Centrality · Importance in a Network

Eigenvector centrality assigns each node a score equal to the weighted sum of its neighbors' scores. It satisfies:

$$x_i = \frac{1}{\lambda} \sum_j A_{ij} x_j$$

Or in matrix form: $A x = \lambda x$ — the **leading eigenvector** of $A$.

This is the basis of PageRank and is used in BorderWatch's Stage 13 (Network Analysis) to identify the most-connected phones.

### Part (a) — by hand

Same 4-node adjacency from Exercise 6:

$$A = \begin{pmatrix}
0 & 1 & 1 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
1 & 0 & 0 & 0
\end{pmatrix}$$

Note: this $A$ is **not symmetric** (directed graph). For undirected analysis, use $A + A^T$.

**Tasks:**

1. Compute the symmetric matrix $B = A + A^T$.
2. Find the leading eigenvector of $B$ (the eigenvector with the largest eigenvalue).
3. Interpret: which phone is most "central"?

### Part (b) — code on all data

Compute eigenvector centrality for the burners-only subgraph. Find the top 5 phones with the highest centrality.


### ✏️ Hand Solution

$$A = \begin{pmatrix}
0 & 1 & 1 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 \\
1 & 0 & 0 & 0
\end{pmatrix}, \quad A^T = \begin{pmatrix}
0 & 1 & 0 & 1 \\
1 & 0 & 0 & 0 \\
1 & 0 & 0 & 0 \\
0 & 0 & 0 & 0
\end{pmatrix}$$

**Step 1 — Symmetric matrix $B = A + A^T$:**

$$B = \begin{pmatrix}
0 & 2 & 1 & 1 \\
2 & 0 & 0 & 0 \\
1 & 0 & 0 & 0 \\
1 & 0 & 0 & 0
\end{pmatrix}$$

**Step 2 — Find the leading eigenvalue/vector** (using power-iteration intuition):

For the leading eigenvalue, multiply $B$ by a vector and observe how it dominates.

Start with $v^{(0)} = [1, 1, 1, 1]^T$:

$B v^{(0)} = [0+2+1+1, 2+0+0+0, 1+0+0+0, 1+0+0+0] = [4, 2, 1, 1]$

Normalize: $\|v\| \approx \sqrt{16+4+1+1} = \sqrt{22} \approx 4.69$, $v^{(1)} \approx [0.853, 0.426, 0.213, 0.213]$.

Iterate once more: $B v^{(1)} = [2(0.426)+0.213+0.213,\ 2(0.853),\ 0.853,\ 0.853] = [1.278, 1.706, 0.853, 0.853]$

This converges roughly to $v \approx [0.85, 0.43, 0.21, 0.21]$ with $\lambda \approx 2.3$.

**Step 3 — Interpretation:**

Phone A has the highest centrality (0.85). It's connected to all the others (degree=3 in $B$). The other three are connected only to A.

> 💡 **The lesson:** eigenvector centrality assigns "importance" recursively — A is important because it connects to many neighbors, who themselves are important.


In [ ]:


print('Computing eigenvector centrality on the burners-only subgraph...')


B = A_sub + A_sub.T


vals, vecs = eig(B.astype(float))
vals = vals.real
vecs = vecs.real
idx_top = np.argmax(vals)
lam_top = vals[idx_top]
centrality = np.abs(vecs[:, idx_top])
centrality = centrality / centrality.sum()

print(f'  Largest eigenvalue: λ = {lam_top:.4f}')
print(f'  Eigenvector shape: {centrality.shape}')


cent_df = pd.DataFrame({
    'phone': sub_phones,
    'centrality': centrality,
    'degree': B.sum(axis=1),
}).sort_values('centrality', ascending=False)

print('\nTop 5 phones by eigenvector centrality:')
print(cent_df.head(5).to_string(index=False))


fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(cent_df['degree'], cent_df['centrality'],
           s=20, alpha=0.5, c='#1d3557')
ax.set_xlabel('Degree (# neighbors)')
ax.set_ylabel('Eigenvector Centrality')
ax.set_title(f'Centrality vs Degree (subgraph of {n_sub} burners)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nThis is the core algorithm behind Stage 13 (SNA) in BorderWatch.')
print('Cell 75 of Mexico_Cartel_DataSet uses nx.eigenvector_centrality(G).')


## Exercise 16 — Projection · Z-Values for Suspicion Score

Projection of a vector $x$ onto a unit vector $v$:

$$\text{proj}_v(x) = (x \cdot v) v$$

The scalar **projection** (the Z-value) is just $z = x \cdot v$.

In BorderWatch, projecting a phone vector onto PC1 gives its "PC1 score" — a 1D summary of all 7 features.

### Part (a) — by hand

A burner with 4 features:

$$x = [117, 0.769, 0.299, 863.7]$$

And a unit vector (already normalized — for the exercise we'll pretend it's PC1):

$$v = [0.20, 0.30, 0.40, 0.85]$$  (we'll verify $\|v\| = 1$ below)

Verify: $\|v\| = \sqrt{0.04 + 0.09 + 0.16 + 0.7225} = \sqrt{1.0125} \approx 1.006$ — close enough to 1 for this exercise.

**Tasks:**

1. Compute the scalar projection $z = x \cdot v$.
2. Compute the vector projection $\text{proj}_v(x) = z \cdot v$.
3. Compute the residual $r = x - \text{proj}_v(x)$. Verify $r \cdot v \approx 0$ (orthogonal).

### Part (b) — code on all data

Project **all 3,734 phones** onto PC1 (the leading eigenvector from Exercise 13). The Z-values are then a single number per phone.


### ✏️ Hand Solution

$$x = [117, 0.769, 0.299, 863.7], \quad v = [0.20, 0.30, 0.40, 0.85]$$

**Step 1 — Scalar projection $z = x \cdot v$:**

$$z = (117)(0.20) + (0.769)(0.30) + (0.299)(0.40) + (863.7)(0.85)$$

$$= 23.4 + 0.231 + 0.120 + 734.145 = \boxed{757.896}$$

**Step 2 — Vector projection $\text{proj}_v(x) = z \cdot v$:**

$$\text{proj}_v(x) = 757.896 \cdot [0.20, 0.30, 0.40, 0.85]$$

$$= [151.58, 227.37, 303.16, 644.21]$$

**Step 3 — Residual $r = x - \text{proj}_v(x)$:**

$$r = [117 - 151.58, 0.769 - 227.37, 0.299 - 303.16, 863.7 - 644.21]$$

$$= [-34.58, -226.60, -302.86, 219.49]$$

(In a "proper" projection with truly unit-norm $v$, the residual would be exactly orthogonal to $v$. The slight numerical drift here is from $\|v\| \approx 1.006$ rather than exactly 1.)

> 💡 **The lesson:** projection is the geometric meaning of PCA. The scalar projection is the "1D score" — the projected phone on a line through the origin.


In [ ]:


print('Projecting all 3,734 phones onto PC1...')


pc1 = eigvecs_sorted[:, 0]
pc1 = pc1 / np.linalg.norm(pc1)
print(f'PC1 unit vector: {np.round(pc1, 4)}')
print(f'Norm of PC1: {np.linalg.norm(pc1):.6f}  (should be 1.0)')


z_pc1 = X_centered @ pc1


_mfg_agg['z_pc1'] = z_pc1

print(f'\nZ-value statistics:')
print(f'  Mean:  {z_pc1.mean():.4f}  (should be ~0 after centering)')
print(f'  Std:   {z_pc1.std():.4f}')
print(f'  Min:   {z_pc1.min():.4f}')
print(f'  Max:   {z_pc1.max():.4f}')


print('\nTop 5 phones by Z (most "different" along PC1):')
top5_z = _mfg_agg.iloc[np.argsort(-np.abs(z_pc1))[:5]]
print(top5_z[['caller_phone', 'z_pc1', 'is_burner', 'total_calls',
              'night_ratio']].to_string(index=False))


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(z_pc1, bins=80, color='#1d3557', alpha=0.85, edgecolor='k')
axes[0].axvline(z_pc1.mean(), color='red', linestyle='--', label=f'mean = {z_pc1.mean():.2f}')
axes[0].set_xlabel('Z (projection onto PC1)')
axes[0].set_ylabel('# phones')
axes[0].set_title('Distribution of Z-values on PC1')
axes[0].legend()

axes[1].scatter(z_pc1, _mfg_agg['composite_score'],
                alpha=0.4, s=6, c='#1d3557')
axes[1].set_xlabel('Z (PC1)')
axes[1].set_ylabel('Composite Score')
axes[1].set_title('Z vs Composite Score (cross-reference)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Exercise 17 — SVD · Numerical Stability and Compression

SVD decomposes any matrix as:

$$X = U \Sigma V^T$$

where:
- $U$ is $m \times m$ orthonormal (left singular vectors)
- $\Sigma$ is $m \times n$ diagonal (singular values $\sigma_i \geq 0$)
- $V^T$ is $n \times n$ orthonormal (right singular vectors)

The singular values $\sigma_i$ relate to PCA eigenvalues by $\sigma_i^2 / (n-1) = \lambda_i$.

### Part (a) — by hand

Same centered $X_c$ from Exercise 8:

$$X_c = \begin{pmatrix}
0.022 & 0.044 \\
-0.039 & 0.016 \\
0.018 & -0.059
\end{pmatrix}$$

**Tasks:**

1. From Exercise 8 we have $X_c^T X_c$. Compute the singular values $\sigma_i = \sqrt{2 \lambda_i^C}$ (where the factor 2 comes from $n-1 = 2$).
2. Verify that the singular values match the eigenvalues from Exercise 12 (scaled by $\sqrt{n-1}$).
3. Why is SVD numerically preferred over Eigendecomposition of $X^T X$?

### Part (b) — code on all data

Compute SVD of the full feature matrix. Compare to PCA results from Exercise 13.


### ✏️ Hand Solution

**Step 1 — $X_c^T X_c$ (from Exercise 8):**

The covariance matrix is $C = \frac{1}{2} X_c^T X_c$, i.e. $X_c^T X_c = 2 C$:

$$X_c^T X_c = 2 \cdot \begin{pmatrix} 0.001151 & -0.000337 \\ -0.000337 & 0.002835 \end{pmatrix} = \begin{pmatrix} 0.002303 & -0.000674 \\ -0.000674 & 0.005671 \end{pmatrix}$$

**Step 2 — Singular values:**

Eigenvalues of $X_c^T X_c$ are 2× those of $C$:

$$\lambda^{X^T X} = 2 \lambda^{C} = 2 \cdot [0.00290,\ 0.00109] = [0.00580,\ 0.00218]$$

Singular values are the square roots:

$$\sigma_1 = \sqrt{0.00580} \approx 0.0762$$
$$\sigma_2 = \sqrt{0.00218} \approx 0.0466$$

$$\boxed{\Sigma = \text{diag}(0.0762,\ 0.0466)}$$

**Step 3 — Verify $\sigma_i^2 / (n-1)$ recovers eigenvalues of $C$:**

$$\frac{\sigma_1^2}{n-1} = \frac{0.00580}{2} = 0.00290 \quad \checkmark = \lambda_1^C$$

$$\frac{\sigma_2^2}{n-1} = \frac{0.00218}{2} = 0.00109 \quad \checkmark = \lambda_2^C$$

Therefore: **SVD of centered matrix = Eigendecomposition of $C$, up to scale**. PCA can be done **via SVD** of $X_c$ instead of eigen of $C$.

**Step 4 — Why SVD is numerically preferred:**

1. **Numerical stability**: for matrices with high condition number, computing $X^T X$ "amplifies" rounding errors. SVD operates directly on $X$ and doesn't require computing $X^T X$.

2. **Compression**: SVD breaks the data into 3 pieces $(U, \Sigma, V)$. Keeping only the top $k$ singular values gives the best rank-$k$ approximation to $X$ (Eckart-Young theorem).

3. **Non-square matrices**: SVD works on any $m \times n$ matrix, even non-square. Eigendecomposition requires square + symmetric.

> 💡 **The critical lesson:** `sklearn.decomposition.PCA` uses SVD internally. When you see `PCA(svd_solver='auto')` — that's it. SVD is the **preferred** way to do PCA.


In [ ]:


print('Computing full SVD on the centered feature matrix...')


U, S_vals, VT = np.linalg.svd(X_centered, full_matrices=False)

print(f'  U shape:  {U.shape}')
print(f'  S shape:  {S_vals.shape}    (singular values, diagonal of Σ)')
print(f'  V^T shape: {VT.shape}')


n = X_centered.shape[0]
eigvals_from_svd = S_vals ** 2 / (n - 1)


print('\nSingular values vs PCA eigenvalues:')
for i in range(len(S_vals)):
    print(f'  σ_{i+1} = {S_vals[i]:.4f},   σ²/(n-1) = {eigvals_from_svd[i]:.4f},   λ_{i+1} from eig(C) = {eigvals_sorted[i]:.4f}')


X_reconstructed = U @ np.diag(S_vals) @ VT
err = np.linalg.norm(X_centered - X_reconstructed)
print(f'\nReconstruction error ||X - UΣV^T||: {err:.4e}  (should be ≈ 0)')


k = 3
X_low_rank = U[:, :k] @ np.diag(S_vals[:k]) @ VT[:k, :]
err_low = np.linalg.norm(X_centered - X_low_rank) / np.linalg.norm(X_centered)
print(f'\nRank-{k} approximation error: {err_low*100:.2f}%  (relative)')
print(f'EVR captured by k={k}: {(S_vals[:k]**2).sum() / (S_vals**2).sum() * 100:.2f}%')


pca_svd = PCA(n_components=7, svd_solver='full')
X_pca_via_svd = pca_svd.fit_transform(X_centered)

diff = np.abs(X_pca_via_svd) - np.abs(X_pca_manual)
print(f'\nDifference |sklearn PCA| - |manual PCA|: max = {np.abs(diff).max():.6e}')
print('  ✓ sklearn PCA uses SVD internally — same result up to sign of components.')


---

# 📙 Part D — End-to-End Synthesis

A single exercise that combines every tool from Sessions 10-11 into the full BorderWatch processing pipeline.


## Exercise 18 — Composite Score Pipeline · Replicating Cell 80

The full Stage 9 pipeline (Cell 80 in Mexico_Cartel_DataSet) combines every tool we've learned:

1. **Build features** (one vector per phone)
2. **StandardScaler** (normalization)
3. **PCA** (dimension reduction)
4. **K-Means** (clustering)
5. **Composite Score** (dot product with expert weights)
6. **Ranking** (top suspects)

### Part (a) — by hand

Compute the Composite Score of 3 burners with the real Cell-80 weights:

$$w = [30,\ 15,\ 20,\ 10,\ 10,\ 10,\ 5]$$

In order (binary signals normalized to [0,1]):
`[is_burner, is_op, has_chain, null_sim, sim_swap, blackout, is_night]`

3 burners with signals (after binarization):

| Phone | is_burner | is_op | has_chain | null_sim | sim_swap | blackout | is_night |
|---|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Phone 1 | 1 | 0 | 1 | 0 | 1 | 0 | 1 |
| Phone 2 | 1 | 1 | 0 | 0 | 0 | 1 | 1 |
| Phone 3 | 1 | 0 | 1 | 1 | 1 | 1 | 1 |

**Tasks:**

1. Compute $\text{score}_i = w \cdot x_i$ for each of the 3 phones.
2. What is the maximum possible score? (all signals = 1)
3. Rank the 3 phones from most to least suspicious.

### Part (b) — code on all data

Run the full pipeline on L1: build signals, compute scores, rank top 10 suspects.


### ✏️ Hand Solution

**Step 1 — Compute score for each phone:**

Form: $\text{score} = \sum_i w_i x_i$.

**Phone 1**: $[1, 0, 1, 0, 1, 0, 1]$

$$\text{score}_1 = 30(1) + 15(0) + 20(1) + 10(0) + 10(1) + 10(0) + 5(1)$$
$$= 30 + 0 + 20 + 0 + 10 + 0 + 5 = \boxed{65}$$

**Phone 2**: $[1, 1, 0, 0, 0, 1, 1]$

$$\text{score}_2 = 30(1) + 15(1) + 20(0) + 10(0) + 10(0) + 10(1) + 5(1)$$
$$= 30 + 15 + 0 + 0 + 0 + 10 + 5 = \boxed{60}$$

**Phone 3**: $[1, 0, 1, 1, 1, 1, 1]$

$$\text{score}_3 = 30(1) + 15(0) + 20(1) + 10(1) + 10(1) + 10(1) + 5(1)$$
$$= 30 + 0 + 20 + 10 + 10 + 10 + 5 = \boxed{85}$$

**Step 2 — Maximum possible:**

$$\text{MAX} = w \cdot \mathbf{1} = 30 + 15 + 20 + 10 + 10 + 10 + 5 = \boxed{100}$$

(The weights sum to 100 — and that's why Cell 80 uses values that sum to 100, so the Composite Score is conveniently interpretable as a "suspicion percentage".)

**Step 3 — Ranking:**

| Phone | Score | % of MAX |
|---|:---:|:---:|
| Phone 3 | 85 | 85% |
| Phone 1 | 65 | 65% |
| Phone 2 | 60 | 60% |

$$\text{Phone 3} > \text{Phone 1} > \text{Phone 2}$$

Phone 3 is **the most suspicious** — it's the only one with 6 of 7 signals, including the critical pair `null_sim` and `sim_swap` together.

> 💡 **The final lesson:** every concept from Sessions 10-11 converges here:
> - **Vectors** = phone profile
> - **Weights** = vector $w$
> - **Dot product** = score
> - **Norms** = distance from "absolute bad" $\mathbf{1}$
> - **PCA** = dimension reduction before this
> - **K-Means** = group assignment
>
> This is why linear algebra is the **core** of data science.


In [ ]:


print('Replicating the full Cell 80 pipeline on all 3,734 phones...')


w_composite = np.array([30, 15, 20, 10, 10, 10, 5])
signal_names = ['is_burner', 'is_op', 'has_chain', 'null_sim',
                'sim_swap', 'blackout', 'is_night']


signals = pd.DataFrame(index=_mfg_agg.index)
signals['is_burner']  = _mfg_agg['is_burner']
signals['is_op']      = (_mfg_agg['total_calls'] > _mfg_agg['total_calls'].quantile(0.85)).astype(int)
signals['has_chain']  = (_mfg_agg['sim_swap_count'] > 1).astype(int)
signals['null_sim']   = (_mfg_agg['null_sim_rate'] > 0.10).astype(int)
signals['sim_swap']   = (_mfg_agg['sim_swap_count'] > 1).astype(int)
signals['blackout']   = (_mfg_agg['weekend_ratio'] < 0.05).astype(int)
signals['is_night']   = (_mfg_agg['night_ratio'] > 0.50).astype(int)


X_signals = signals[signal_names].values
composite_scores = X_signals @ w_composite

_mfg_agg['composite_score_v2'] = composite_scores

print(f'\nFinal composite scores statistics:')
print(f'  Min:   {composite_scores.min()}')
print(f'  Max:   {composite_scores.max()}')
print(f'  Mean:  {composite_scores.mean():.2f}')
print(f'  ≥ 50:  {(composite_scores >= 50).sum()} phones  (HIGH suspicion)')
print(f'  ≥ 75:  {(composite_scores >= 75).sum()} phones  (CRITICAL suspicion)')


top10 = _mfg_agg.sort_values('composite_score_v2', ascending=False).head(10)
print('\nTop 10 suspects (full pipeline):')
display_cols = ['caller_phone', 'composite_score_v2', 'is_burner',
                'total_calls', 'night_ratio', 'null_sim_rate', 'sim_swap_count']
print(top10[display_cols].to_string(index=False))


fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(composite_scores, bins=50, color='#1d3557', alpha=0.85, edgecolor='k')
axes[0].axvline(50, color='orange', linestyle='--', label='HIGH (≥50)')
axes[0].axvline(75, color='red', linestyle='--', label='CRITICAL (≥75)')
axes[0].set_xlabel('Composite Score (0-100)')
axes[0].set_ylabel('# phones')
axes[0].set_title('Score distribution (full pipeline)')
axes[0].legend()


signal_counts = signals[signal_names].sum().sort_values(ascending=True)
axes[1].barh(range(len(signal_counts)), signal_counts.values,
             color='#E63946', edgecolor='k', alpha=0.85)
axes[1].set_yticks(range(len(signal_counts)))
axes[1].set_yticklabels(signal_counts.index, fontsize=9)
axes[1].set_xlabel('# phones triggering signal')
axes[1].set_title('Signal trigger counts')
plt.tight_layout()
plt.show()


---

# 🎓 Summary

We covered **18 exercises** that span every linear-algebra concept from Sessions 10-11:

| Part | Exercise | Concept | BorderWatch Application |
|:---:|:---:|---|---|
| A | 1 | Vector operations | Average burner profile |
| A | 2 | Dot product | Composite Score |
| A | 3 | Linear combination | Building a weighted feature |
| A | 4 | Norms (L1, L2, L∞) | Distances, outliers |
| A | 5 | Cosine similarity | Pattern Mirroring (Part 16) |
| B | 6 | Adjacency matrix | Burner detection (degree=1) |
| B | 7 | A² multiplication | Chain Detection (Part 14) |
| B | 8 | Transpose + Trace | Covariance Matrix |
| B | 9 | Rank | Multicollinearity |
| B | 10 | Determinant | Volume of feature space |
| B | 11 | Matrix inverse | Mahalanobis distance |
| C | 12 | Eigenvalues 2×2 | Variance per axis |
| C | 13 | PCA from scratch | Part 21 — dimension reduction |
| C | 14 | EVR | Choosing optimal k |
| C | 15 | Eigenvector centrality | NetworkX (Part 13) |
| C | 16 | Projection | Z values |
| C | 17 | SVD | Numerical stability |
| D | 18 | End-to-End | Stage 9 — Composite Score Pipeline |

#### 🔑 The Unifying Principle

**Everything we did is three basic operations:**

1. **Dot product** ($w \cdot x$) → score, regression, SHAP
2. **Matrix multiplication** ($A \cdot B$) → graphs, projections, transformations
3. **Eigendecomposition** → PCA, centrality, stability

This is the core. Every pipeline in BorderWatch — from Stage 1 (Burner Detection) to Stage 21 (PCA + K-Means) — is built from these three operations.

---
